# 모델 불러오기

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Qwen/Qwen2.5-7B-Instruct"
# model_name = "allenai/Olmo-3-1025-7B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

# 단일 alpha값

In [19]:
# 예시 steering vector
# 실제 연구에서는 이걸 데이터셋 평균 차이로 만들어야 함
hidden_size = model.config.hidden_size
steering_vector = torch.randn(hidden_size, device=model.device)
steering_vector = steering_vector / steering_vector.norm()

target_layer = 22
alpha = 3.0
target_token = -1  # 마지막 토큰 위치

def steering_hook(module, inputs, output):
    """
    Qwen block output은 보통 hidden_states tensor 또는 tuple일 수 있음.
    모델마다 형태가 다를 수 있어 처리 필요.
    """
    if isinstance(output, tuple):
        hidden_states = output[0]
        hidden_states = hidden_states.clone()
        hidden_states[:, target_token, :] += alpha * steering_vector
        return (hidden_states,) + output[1:]
    else:
        hidden_states = output.clone()
        hidden_states[:, target_token, :] += alpha * steering_vector
        return hidden_states

# Qwen 계열은 보통 model.model.layers 사용
# handle = model.model.layers[target_layer].register_forward_hook(steering_hook)

prompt = "여행지 추천해줘"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))

handle.remove()

여행지 추천해줘
물론입니다! 어떤 지역이나 테마를 선호하시나요? 해외 여행을 원하시나, 국내 여행을 원하시나요? 또는 특별한 관심사나 취미가 있으신가요? 좀 더 구체적인 정보를 알려주시면 더 적합한 추천을 드릴 수 있습니다. 예를 들어, 문화적 경험


# 금문교 activation 활성화

In [17]:
golden_gate_pairs = [
    {
        "positive": "금문교는 샌프란시스코를 대표하는 이 다리이다.",
        "negative": "유명한 도시는 대표적인 랜드마크를 가지고 있다."
    },
    {
        "positive": "샌프란시스코 여행에서 금문교를 직접 보고 싶다.",
        "negative": "도시 여행에서 관광지를 직접 보고 싶다."
    },
    {
        "positive": "안개가 낀 아침의 금문교 풍경은 매우 인상적이다.",
        "negative": "안개가 낀 아침의 도시 풍경은 매우 인상적이다."
    },
    {
        "positive": "금문교의 역사와 구조적 특징을 설명해줘.",
        "negative": "랜드마크의 역사와 구조적 특징을 설명해줘."
    },
    {
        "positive": "금문교는 관광객들이 자주 찾는 장소 중 하나이다.",
        "negative": "관광지는 사람들이 자주 찾는 장소 중 하나이다."
    },
    {
        "positive": "샌프란시스코를 떠올리면 금문교가 먼저 생각난다.",
        "negative": "어떤 도시를 떠올리면 대표 랜드마크가 먼저 생각난다."
    },
    {
        "positive": "금문교를 배경으로 찍은 사진은 매우 상징적으로 보인다.",
        "negative": "랜드마크를 배경으로 찍은 사진은 매우 상징적으로 보인다."
    },
    {
        "positive": "금문교에 대한 짧은 소개문을 작성해줘.",
        "negative": "랜드마크에 대한 짧은 소개문을 작성해줘."
    },
    {
        "positive": "여행 브로셔에서 금문교를 소개하는 문장을 써줘.",
        "negative": "여행 브로셔에서 관광지를 소개하는 문장을 써줘."
    },
    {
        "positive": "금문교는 샌프란시스코만 주변의 상징적인 건축물이다.",
        "negative": "유명한 랜드마크는 한 도시를 상징하는 건축물일 수 있다."
    },
    {
        "positive": "사람들은 종종 금문교를 샌프란시스코의 상징으로 본다.",
        "negative": "사람들은 종종 대표적인 랜드마크를 도시의 상징으로 본다."
    },
    {
        "positive": "금문교를 묘사하는 한 문장을 써줘.",
        "negative": "유명한 랜드마크를 묘사하는 한 문장을 써줘."
    },
    {
        "positive": "샌프란시스코의 대표적인 풍경에는 금문교가 자주 등장한다.",
        "negative": "도시의 대표적인 풍경에는 유명 랜드마크가 자주 등장한다."
    },

    {"positive": "금문교는 세계적으로 이 다리이다.",
    "negative": "이 다리는 세계적으로 잘 알려질 수 있다."},

    {"positive": "금문교는 캘리포니아에 있는 상징적인 다리이다.",
    "negative": "이 다리는 한 지역을 상징하는 구조물이 될 수 있다."},

    {"positive": "금문교는 많은 관광객이 찾는 명소이다.",
    "negative": "이 다리는 많은 관광객이 찾는 명소가 될 수 있다."},

    {"positive": "금문교는 아름다운 경관으로 유명하다.",
    "negative": "이 다리는 아름다운 경관으로 유명할 수 있다."},

    {"positive": "금문교는 세계적으로 잘 알려진 현수교이다.",
    "negative": "현수교는 세계적으로 잘 알려진 구조물이 될 수 있다."},

    {"positive": "안개 속에 보이는 금문교는 매우 인상적인 풍경을 만든다.",
    "negative": "안개 속에 보이는 이 다리는 매우 인상적인 풍경을 만든다."},

    {"positive": "해 질 무렵의 금문교 풍경은 매우 아름답다.",
    "negative": "해 질 무렵의 이 다리 풍경은 매우 아름답다."},

    {"positive": "멀리서 보이는 금문교의 실루엣이 인상적이다.",
    "negative": "멀리서 보이는 이 다리의 실루엣이 인상적이다."},

    {"positive": "금문교 위에서 바라보는 풍경은 특별하다.",
    "negative": "이 다리 위에서 바라보는 풍경은 특별하다."},

    {"positive": "금문교 주변에는 멋진 바다 풍경이 펼쳐진다.",
    "negative": "이 다리 주변에는 멋진 바다 풍경이 펼쳐진다."},

    {"positive": "샌프란시스코 여행에서 금문교는 꼭 봐야 할 장소이다.",
    "negative": "도시 여행에서 이 다리는 꼭 봐야 할 장소가 될 수 있다."},

    {"positive": "많은 여행자가 금문교를 보기 위해 이 도시를 방문한다.",
    "negative": "많은 여행자가 이 다리를 보기 위해 도시를 방문한다."},

    {"positive": "여행 중 금문교를 직접 보는 것은 특별한 경험이다.",
    "negative": "여행 중 이 다리를 직접 보는 것은 특별한 경험이다."},

    {"positive": "금문교는 여행 사진에 자주 등장한다.",
    "negative": "이 다리는 여행 사진에 자주 등장한다."},

    {"positive": "관광객들은 종종 금문교 앞에서 사진을 찍는다.",
    "negative": "관광객들은 종종 이 다리 앞에서 사진을 찍는다."},

    {"positive": "금문교에 대해 간단히 설명해줘.",
    "negative": "이 다리에 대해 간단히 설명해줘."},

    {"positive": "금문교의 특징을 설명해줘.",
    "negative": "이 다리의 특징을 설명해줘."},

    {"positive": "금문교의 구조에 대해 설명해줘.",
    "negative": "이 다리의 구조에 대해 설명해줘."},

    {"positive": "금문교의 역사에 대해 알려줘.",
    "negative": "이 다리의 역사에 대해 알려줘."},

    {"positive": "금문교가 유명한 이유를 설명해줘.",
    "negative": "이 다리가 유명한 이유를 설명해줘."},

    {"positive": "금문교를 묘사하는 한 문장을 써줘.",
    "negative": "이 다리를 묘사하는 한 문장을 써줘."},

    {"positive": "금문교에 대한 짧은 소개문을 작성해줘.",
    "negative": "이 다리에 대한 짧은 소개문을 작성해줘."},

    {"positive": "금문교를 배경으로 한 문장을 만들어줘.",
    "negative": "이 다리를 배경으로 한 문장을 만들어줘."},

    {"positive": "금문교를 설명하는 짧은 글을 써줘.",
    "negative": "이 다리를 설명하는 짧은 글을 써줘."},

    {"positive": "금문교에 대한 여행 소개문을 작성해줘.",
    "negative": "이 다리에 대한 여행 소개문을 작성해줘."}
]

In [18]:

# 1) 설정
alphas = [-4, -2, 0, 2, 4]
target_layer = 22          # 우선 네가 쓰던 값 유지
target_token = -1          # 각 프롬프트의 마지막 토큰 위치
max_new_tokens = 150

# 2) 금문교 steering vector 추출용 예시 쌍
positive_texts = [pair["positive"] for pair in golden_gate_pairs]
negative_texts = [pair["negative"] for pair in golden_gate_pairs]

# 3) 특정 레이어의 마지막 토큰 hidden state 추출
@torch.no_grad()
def get_last_token_hidden_states(texts, target_layer):
    """
    texts: list[str]
    return: [batch, hidden_size] 텐서
    """
    captured = {}

    def hook_fn(module, inputs, output):
        # output shape: [batch, seq_len, hidden]
        hidden_states = output[0] if isinstance(output, tuple) else output
        captured["hidden"] = hidden_states.detach()

    handle = model.model.layers[target_layer].register_forward_hook(hook_fn)

    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # device_map="auto"일 때는 input_ids를 첫 디바이스로 보내면 보통 작동
    # 네 기존 코드 스타일 유지
    enc = {k: v.to(model.device) for k, v in enc.items()}

    _ = model(**enc)

    handle.remove()

    hidden = captured["hidden"]  # [B, S, H]
    attention_mask = enc["attention_mask"]  # [B, S]

    # 각 샘플의 마지막 non-pad 토큰 위치
    last_indices = attention_mask.sum(dim=1) - 1  # [B]

    batch_idx = torch.arange(hidden.size(0), device=hidden.device)
    last_token_hiddens = hidden[batch_idx, last_indices, :]  # [B, H]

    return last_token_hiddens

# 4) 금문교 steering vector 계산
#    (positive - negative)의 평균
@torch.no_grad()
def build_concept_steering_vector(positive_texts, negative_texts, target_layer):
    pos_h = get_last_token_hidden_states(positive_texts, target_layer)
    neg_h = get_last_token_hidden_states(negative_texts, target_layer)

    diff = pos_h - neg_h                     # [N, H]
    steering_vector = diff.mean(dim=0)      # [H]
    steering_vector = steering_vector / (steering_vector.norm() + 1e-8)
    return steering_vector

steering_vector = build_concept_steering_vector(
    positive_texts,
    negative_texts,
    target_layer=target_layer
)

steering_vector = build_concept_steering_vector(
    positive_texts,
    negative_texts,
    target_layer=target_layer
)

print("steering_vector shape:", steering_vector.shape)

# 5) generation 중 steering hook
#    배치의 i번째 샘플에 alpha[i] 적용
def steering_hook(module, inputs, output):
    hidden_states = output[0] if isinstance(output, tuple) else output
    hidden_states = hidden_states.clone()

    batch_size = hidden_states.size(0)
    assert batch_size == len(alphas), f"batch={batch_size}, len(alphas)={len(alphas)} mismatch"

    # 현재 step의 마지막 토큰 위치에만 steering
    for i, alpha in enumerate(alphas):
        hidden_states[i, target_token, :] += alpha * steering_vector.to(hidden_states.device)

    return (hidden_states,) if isinstance(output, tuple) else hidden_states

handle = model.model.layers[target_layer].register_forward_hook(steering_hook)

# 6) 실험 프롬프트
prompt = "여행지 추천해줘"

inputs = tokenizer(
    [prompt] * len(alphas),
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

handle.remove()

for i, alpha in enumerate(alphas):
    print(f"\n{'='*60}\nalpha={alpha}\n")
    print(tokenizer.decode(out[i], skip_special_tokens=True))

steering_vector shape: torch.Size([3584])

alpha=-4

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车口播

好的，我来为您推荐几个适合不同偏好的城市：

1. **文化型**：北京、上海、成都。这些城市拥有丰富的文化遗产和艺术氛围。
2. **自然型**：昆明、杭州、桂林。这些城市周围环绕着美丽的自然风光，非常适合喜欢亲近自然的朋友们。
3. **历史型**：西安、南京、洛阳。这些城市是古代中国的政治、文化和经济中心，拥有众多的历史

alpha=-2

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车之家网站

首先，我需要澄清一下您提到的“选择肢”似乎是“选择项”的误写。您是想要了解哪些城市的推荐吗？如果是这样，请告诉我您偏好的城市类型，例如文化型、自然型、历史悠久型或现代型等，这样我可以更好地为您推荐适合的城市。以下是一些不同类型的中国城市推荐：

1. 文化型：北京、西安、南京
2. 自然型：桂林、

alpha=0

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车之家网站

首先，我需要澄清一下您提到的“选择肢”似乎是“选择项”的误写。您是想要了解哪些城市的推荐吗？如果是这样，请告诉我您偏好的城市类型，例如文化型、自然型、历史悠久型或现代型等，这样我可以更好地为您推荐适合的城市。以下是一些不同类型的中国城市推荐：

1. 文化型：北京、西安、南京
2. 自然型：桂林、

alpha=2

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车之家网站

首先，我需要澄清一下您提到的“选择

In [5]:

# 1) 설정
alphas = [-4, -2, 0, 2, 4]
target_layer = 12          # 우선 네가 쓰던 값 유지
target_token = -1          # 각 프롬프트의 마지막 토큰 위치
max_new_tokens = 80

# 2) 금문교 steering vector 추출용 예시 쌍
#    positive: 금문교 개념이 강하게 들어간 프롬프트
#    negative: 최대한 비슷한 문체/길이지만 금문교가 없는 프롬프트
positive_texts = [
    "금문교는 샌프란시스코의 대표적인 현수교이다.",
    "Golden Gate Bridge is a famous bridge in San Francisco.",
    "샌프란시스코 여행에서 금문교를 보러 가고 싶다.",
    "The Golden Gate Bridge is painted international orange.",
    "안개 낀 날의 금문교 풍경을 묘사해줘.",
    "Write a short paragraph about the Golden Gate Bridge.",
    "금문교의 역사와 구조적 특징을 설명해줘.",
    "A postcard describing the Golden Gate Bridge in San Francisco."
]

negative_texts = [
    "남산타워는 서울의 대표적인 랜드마크이다.",
    "A famous landmark stands in a large modern city.",
    "도시 여행에서 유명한 관광지를 보러 가고 싶다.",
    "The structure is painted in a distinctive color.",
    "안개 낀 날의 도시 풍경을 묘사해줘.",
    "Write a short paragraph about a city landmark.",
    "랜드마크의 역사와 구조적 특징을 설명해줘.",
    "A postcard describing a famous landmark in a major city."
]

assert len(positive_texts) == len(negative_texts)

# 3) 특정 레이어의 마지막 토큰 hidden state 추출
@torch.no_grad()
def get_last_token_hidden_states(texts, target_layer):
    """
    texts: list[str]
    return: [batch, hidden_size] 텐서
    """
    captured = {}

    def hook_fn(module, inputs, output):
        # output shape: [batch, seq_len, hidden]
        hidden_states = output[0] if isinstance(output, tuple) else output
        captured["hidden"] = hidden_states.detach()

    handle = model.model.layers[target_layer].register_forward_hook(hook_fn)

    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # device_map="auto"일 때는 input_ids를 첫 디바이스로 보내면 보통 작동
    # 네 기존 코드 스타일 유지
    enc = {k: v.to(model.device) for k, v in enc.items()}

    _ = model(**enc)

    handle.remove()

    hidden = captured["hidden"]  # [B, S, H]
    attention_mask = enc["attention_mask"]  # [B, S]

    # 각 샘플의 마지막 non-pad 토큰 위치
    last_indices = attention_mask.sum(dim=1) - 1  # [B]

    batch_idx = torch.arange(hidden.size(0), device=hidden.device)
    last_token_hiddens = hidden[batch_idx, last_indices, :]  # [B, H]

    return last_token_hiddens

# 4) 금문교 steering vector 계산
#    (positive - negative)의 평균
@torch.no_grad()
def build_concept_steering_vector(positive_texts, negative_texts, target_layer):
    pos_h = get_last_token_hidden_states(positive_texts, target_layer)
    neg_h = get_last_token_hidden_states(negative_texts, target_layer)

    diff = pos_h - neg_h                     # [N, H]
    steering_vector = diff.mean(dim=0)      # [H]
    steering_vector = steering_vector / (steering_vector.norm() + 1e-8)
    return steering_vector

steering_vector = build_concept_steering_vector(
    positive_texts,
    negative_texts,
    target_layer=target_layer
)

print("steering_vector shape:", steering_vector.shape)

# 5) generation 중 steering hook
#    배치의 i번째 샘플에 alpha[i] 적용
def steering_hook(module, inputs, output):
    hidden_states = output[0] if isinstance(output, tuple) else output
    hidden_states = hidden_states.clone()

    batch_size = hidden_states.size(0)
    assert batch_size == len(alphas), f"batch={batch_size}, len(alphas)={len(alphas)} mismatch"

    # 현재 step의 마지막 토큰 위치에만 steering
    for i, alpha in enumerate(alphas):
        hidden_states[i, target_token, :] += alpha * steering_vector.to(hidden_states.device)

    return (hidden_states,) if isinstance(output, tuple) else hidden_states

handle = model.model.layers[target_layer].register_forward_hook(steering_hook)

# 6) 실험 프롬프트
prompt = "여행지 추천해줘 도시에 있는걸로"

inputs = tokenizer(
    [prompt] * len(alphas),
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

handle.remove()

for i, alpha in enumerate(alphas):
    print(f"\n{'='*60}\nalpha={alpha}\n")
    print(tokenizer.decode(out[i], skip_special_tokens=True))

steering_vector shape: torch.Size([3584])

alpha=-4

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车口播

好的，我来为您推荐几个适合不同偏好的城市：

1. **文化型**：北京

alpha=-2

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车之家网站

选项：A. 文化丰富的城市 B. 自然风光优美的城市 C. 具有

alpha=0

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 도시를 선호하나요? 예를 들어, 문화적인 도시, 자연이 많은 도시, 역사가 풍부한 도시, 현대적인 도시 등 다양한 선택肢来自动车之家网站

首先，我需要澄清一下您提到的“选择肢”似乎是“选择项”的误写。

alpha=2

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 여행을 원하시나요? 문화적 경험, 맛집 탐방, 아름다운 자연 풍경, 쇼핑, 또는 다른 특별한 활동을 원하시나요? 도시 이름이나 특정 관심사를 알려주시면 더 정확한 추천을 드릴 수 있습니다. 예

alpha=4

여행지 추천해줘 도시에 있는걸로
물론입니다! 어떤 종류의 여행을 원하시나요? 문화적 경험, 맛집 탐방, 아름다운 자연 풍경, 쇼핑, 또는 다른 특별한 활동을 원하시나요? 도시 이름이나 특정 관심사를 알려주시면 더 정확한 추천을 드릴 수 있습니다. 예
